Generate YouTube-style chapter timestamps for podcast episodes. WhisperX detects where speech starts and stops, then an LLM summarizes each segment into a chapter title.

The end result looks like this:

```
0:00 - Experiencing self vs remembering self
0:10 - Whether memories are the primary source of happiness
0:36 - Controlling how we remember experiences
```

## Problem

You have podcast episodes and want to generate chapter markers — the kind you see in YouTube descriptions or podcast apps. Each chapter needs a timestamp and a short description of what's being discussed.

## Solution

**What's in this recipe:**

1. **WhisperX** transcribes the audio and detects speech boundaries via VAD (Voice Activity Detection)
1. `json.list_iterator` creates a **view** with one row per speech segment
1. **GPT-4o-mini** generates a chapter title for each segment as a computed column
1. The result is formatted as YouTube-style timestamps

### Setup

In [ ]:
%pip install -qU pixeltable whisperx openai

In [1]:
import getpass
import os

if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API Key: ')

In [2]:
import pixeltable as pxt
import pixeltable.functions as pxtf

pxt.drop_dir('podcast_demo', force=True, if_not_exists='ignore')
pxt.create_dir('podcast_demo')

### Load a podcast episode

In [4]:
episodes = pxt.create_table(
    'podcast_demo/episodes', {'title': pxt.String, 'audio': pxt.Audio}
)

Created table 'episodes'.


In [5]:
episodes.insert(
    [
        {
            'title': 'Lex Fridman Podcast Excerpt',
            'audio': 'https://raw.githubusercontent.com/pixeltable/pixeltable/release/docs/resources/audio-transcription-demo/Lex-Fridman-Podcast-430-Excerpt-0.mp4',
        }
    ]
)

Inserted 1 row with 0 errors in 0.82 s (1.22 rows/s)


1 row inserted.

View the data:

In [6]:
episodes.collect()

title,audio
Lex Fridman Podcast Excerpt,


And the table schema so far:

In [7]:
episodes

table 'podcast_demo/episodes'

 Column Name    Type    Source Computed With Comment
----------------------------------------------------
       title  String  episodes                      
       audio   Audio  episodes

### Step 1: Transcribe with WhisperX

WhisperX does two things at once: it runs Voice Activity Detection (VAD) to find where speech occurs, then transcribes each speech region. The output is a list of segments — each with a start time, end time, and the text that was spoken.

These segments are the raw material for our chapter markers. Each segment boundary corresponds to a natural pause or transition in the conversation.

> **Note:** You may see verbose warnings from `torchcodec` or `pyannote` when this cell runs — they're harmless and can be ignored.

In [8]:
episodes.add_computed_column(
    transcription=pxtf.whisperx.transcribe(
        episodes.audio, model='tiny.en'
    )
)

Added 1 column value with 0 errors in 9.81 s (0.10 rows/s)


1 row updated.

In [9]:
episodes.select(
    first_segment=episodes.transcription.segments[0]
).collect()

first_segment
"{""end"": 10.443, ""text"": "" of experiencing self versus remembering self. I was hoping you can give a simple answer of how we should live life."", ""start"": 0.908, ""avg_logprob"": -0.063}"


Each segment is a dict with `start`, `end`, and `text` fields. We can use the `['*']` path expression to extract a field from every segment at once — this returns a list of values:

In [10]:
episodes.select(
    segment_starts=episodes.transcription.segments['*'].start,
    segment_ends=episodes.transcription.segments['*'].end,
    segment_text=episodes.transcription.segments['*'].text,
).collect()

segment_starts,segment_ends,segment_text
"[0.908, 10.73, 36.312]","[10.443, 36.194, 58.891]","["" of experiencing self versus remembering self. I was hoping you can give a simple answer of how we should live life."", "" Based on the fact that our memories could be a source of happiness or could be the primary source of happiness that an event when experienced bea ...... it's remembered over and over and over and over and maybe there is some wisdom and the fact that we can control to some degree how we remember it."", "" how we evolve our memory of it, such that it can maximize the long-term happiness of that repeated experience. Okay, well, first, I'll say, I wis ...... an I be your opening actor? Oh, my God. No, I've got to hope it for you, dude. Otherwise, it's like, you know, everybody leaves after you're done.""]"


That's useful for peeking at the data, but the result is still one row with parallel lists — not one row per segment. Here's what a single segment looks like as a proper row:

In [11]:
episodes.select(
    start=episodes.transcription.segments[0].start,
    end=episodes.transcription.segments[0].end,
    text=episodes.transcription.segments[0].text,
).collect()

start,end,text
0.908,10.443,of experiencing self versus remembering self. I was hoping you can give a simple answer of how we should live life.


We want every segment as its own row like this — not just the first one. That's what `list_iterator` does.

### Step 2: Create a segments view with `list_iterator`

To work with individual segments as rows, we create a **view** using `pxtf.json.list_iterator`. This iterator takes parallel lists and zips them into one row per element — like converting columns of arrays into a proper table.

The keyword argument names (`start`, `end`, `text`) become the column names in the view. Each argument is a Pixeltable expression that evaluates to a JSON list.

**Why `astype`?** WhisperX returns an untyped `dict`, so Pixeltable doesn't know what types are inside the segments. `list_iterator` requires typed JSON so it can define the view's schema. We use `astype` to declare that `.start` and `.end` are lists of floats, and `.text` is a list of strings:

In [12]:
segments = pxt.create_view(
    'podcast_demo/segments',
    episodes,
    iterator=pxtf.json.list_iterator(
        start=episodes.transcription.segments['*'].start.astype(
            pxt.Json[[float]]
        ),
        end=episodes.transcription.segments['*'].end.astype(
            pxt.Json[[float]]
        ),
        text=episodes.transcription.segments['*'].text.astype(
            pxt.Json[[str]]
        ),
    ),
)

Here is the schema of this view - you can see the four columns added by the `list_iterator`, and the other columns that came from the `episodes` table we started with:

In [13]:
segments

view 'podcast_demo/segments' (of 'podcast_demo/episodes')

    Column Name              Type    Source                       Computed With Comment
---------------------------------------------------------------------------------------
            pos     Required[Int]  segments                       list_iterator        
          start   Required[Float]  segments                       list_iterator        
            end   Required[Float]  segments                       list_iterator        
           text  Required[String]  segments                       list_iterator        
.......................................................................................
          title            String  episodes                                            
          audio             Audio  episodes                                            
  transcription              Json  episodes  transcribe(audio, model='tiny.en')

One row per segment, with typed columns. From here we can add computed columns that operate on individual segments — no manual JSON wrangling needed.

In [14]:
segments.select(segments.start, segments.end, segments.text).collect()

start,end,text
0.908,10.443,of experiencing self versus remembering self. I was hoping you can give a simple answer of how we should live life.
10.73,36.194,Based on the fact that our memories could be a source of happiness or could be the primary source of happiness that an event when experienced bears its fruits the most when it's remembered over and over and over and over and maybe there is some wisdom and the fact that we can control to some degree how we remember it.
36.312,58.891,"how we evolve our memory of it, such that it can maximize the long-term happiness of that repeated experience. Okay, well, first, I'll say, I wish I could take you on the road with me. That was such a great description. Can I be your opening actor? Oh, my God. No, I've got to hope it for you, dude. Otherwise, it's like, you know, everybody leaves after you're done."


### Step 3: Generate chapter titles

Each segment now has its own row. We add a computed column that sends the segment text to GPT-4o-mini for a short chapter title:

In [15]:
segments.add_computed_column(
    title_response=pxtf.openai.chat_completions(
        messages=[
            {
                'role': 'user',
                'content': pxtf.string.format(
                    'Write a short chapter title (5-10 words) for this podcast segment. '
                    'Return only the title, no quotes or extra punctuation.\n\n{0}',
                    segments.text,
                ),
            }
        ],
        model='gpt-4o-mini',
    ),
    if_exists='replace',
)

segments.add_computed_column(
    chapter_title=segments.title_response.choices[0].message.content,
    if_exists='replace',
)

Added 3 column values with 0 errors in 2.64 s (1.14 rows/s)
Added 3 column values with 0 errors in 0.02 s (142.43 rows/s)


3 rows updated.

In [16]:
segments.select(segments.text, segments.chapter_title).collect()

text,chapter_title
of experiencing self versus remembering self. I was hoping you can give a simple answer of how we should live life.,Living in the Moment vs. Reflecting on the Past
Based on the fact that our memories could be a source of happiness or could be the primary source of happiness that an event when experienced bears its fruits the most when it's remembered over and over and over and over and maybe there is some wisdom and the fact that we can control to some degree how we remember it.,The Power of Memory in Shaping Happiness
"how we evolve our memory of it, such that it can maximize the long-term happiness of that repeated experience. Okay, well, first, I'll say, I wish I could take you on the road with me. That was such a great description. Can I be your opening actor? Oh, my God. No, I've got to hope it for you, dude. Otherwise, it's like, you know, everybody leaves after you're done.",Evolving Memories for Lasting Happiness


### Step 4: YouTube-style timestamps

Format each segment's start time with its LLM-generated title:

In [17]:
@pxt.udf
def format_timestamp(start: float, chapter_title: str) -> str:
    mins, secs = divmod(int(start), 60)
    title = chapter_title.strip('"')
    return f'{mins}:{secs:02d} - {title}'


segments.add_computed_column(
    timestamp=format_timestamp(segments.start, segments.chapter_title),
    if_exists='replace',
)

Added 3 column values with 0 errors in 0.04 s (69.92 rows/s)


3 rows updated.

In [18]:
segments.select(segments.timestamp).order_by(segments.start).collect()

timestamp
0:00 - Living in the Moment vs. Reflecting on the Past
0:10 - The Power of Memory in Shaping Happiness
0:36 - Evolving Memories for Lasting Happiness


Here is a formatted version you can copy/paste directly into your YouTube video description field:

In [19]:
print('\n'.join(segments.order_by(segments.start).collect()['timestamp']))

0:00 - Living in the Moment vs. Reflecting on the Past
0:10 - The Power of Memory in Shaping Happiness
0:36 - Evolving Memories for Lasting Happiness


### Step 5: Listen to each segment

The segments view inherits the `audio` column from the base `episodes` table. We can add a computed column that slices out each segment's audio clip using its `start` and `end` times — useful for spot-checking the transcription.

In Jupyter, the `audio_clip` column renders as an inline audio player you can click to listen:

In [20]:
import os
import subprocess
import tempfile


@pxt.udf
def slice_audio(audio: pxt.Audio, start: float, end: float) -> pxt.Audio:
    """Extract a time range from an audio file using ffmpeg."""
    fd, output_path = tempfile.mkstemp(suffix='.mp4')
    os.close(fd)
    subprocess.run(
        [
            'ffmpeg',
            '-y',
            '-i',
            str(audio),
            '-ss',
            str(start),
            '-to',
            str(end),
            '-c',
            'copy',
            output_path,
        ],
        capture_output=True,
        check=True,
    )
    return output_path


segments.add_computed_column(
    audio_clip=slice_audio(segments.audio, segments.start, segments.end),
    if_exists='replace',
)

Added 3 column values with 0 errors in 0.28 s (10.85 rows/s)


3 rows updated.

In [21]:
segments.select(segments.timestamp, segments.audio_clip).order_by(
    segments.start
).collect()

timestamp,audio_clip
0:00 - Living in the Moment vs. Reflecting on the Past,
0:10 - The Power of Memory in Shaping Happiness,
0:36 - Evolving Memories for Lasting Happiness,


## Explanation

**Pipeline:**

```
Audio → WhisperX (VAD + transcription) → list_iterator view (1 row per segment) → GPT-4o-mini per row → timestamps
```

The `episodes` table holds the raw audio and transcription. The `segments` view fans out the transcription's segment list into individual rows using `json.list_iterator`. Each segment row then gets its own LLM call and timestamp — all as computed columns.

Insert a new episode and the entire pipeline runs automatically: transcription, segment extraction, chapter titling, and timestamp formatting.

**How WhisperX finds the chapter boundaries:**

WhisperX uses PyAnnote VAD to detect where speech occurs in the audio. Pauses, silence, and transitions between speakers create natural segment boundaries. These boundaries become the chapter start times.

**Why `list_iterator`?**

Without `list_iterator`, you'd write custom UDFs to extract segments from the JSON, batch them into a single LLM prompt, and parse the response back apart. The view-based approach is more idiomatic — each segment is its own row, and computed columns operate on one row at a time.

**Trade-offs:**

- One LLM call per segment instead of one batched call — fine for short podcasts, but consider batching for episodes with 50+ segments
- This approach requires running a full transcription model just to find where the pauses are
- For longer episodes, WhisperX's `chunk_size` parameter controls how the audio is batched internally
- The chapter titles depend on LLM quality — `gpt-4o-mini` is fast and cheap, use `gpt-4o` for higher quality

## See also

- [`json.list_iterator`](https://docs.pixeltable.com/sdk/latest/json#iterator-list_iterator) — Iterator for flattening JSON lists into view rows
- [Transcribe audio](https://docs.pixeltable.com/howto/cookbooks/audio/audio-transcribe) — Fixed-interval splitting with local Whisper
- [Summarize podcasts](https://docs.pixeltable.com/howto/cookbooks/audio/audio-summarize-podcast) — Transcription + LLM summarization
- [Video scene detection](https://docs.pixeltable.com/howto/cookbooks/video/video-scene-detection) — Content-aware detection for video